# Day 1 Project — Smart Research Assistant

Everything from today, assembled:

```text
model call -> structured tool request -> validated Python tool -> observation
           -> bounded agent loop -> validated final response
```

A demo that works once is not evidence. We finish by running a small behaviour suite and
then deliberately triggering the three failures this system is designed to survive.

## Before you begin

### Learning outcomes

- Run the finished assistant and read status, steps, tools used and token usage.
- Run a four-case behaviour suite and check the tools against expectations.
- Trigger three failures on purpose and name the layer each one belongs to.

Architecture reference: [D01–D05](../../diagrams/source/day_01.md).

### Expected observation

The suite completes in MOCK mode with no API credit spent. The failure demonstrations
print `failed` statuses with specific error messages — never a traceback.

## Concept briefing

## What to carry into Day 2

Day 1 creates a bounded model-and-tool system, but the model still relies on information
inside its request or learned during training. Day 2 introduces external knowledge. The
agent loop remains the same; the new question is how to retrieve the right evidence and
prove the answer used it.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/research_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

### Step 1 — Assemble the project

Mock is the default. It is not a lesser mode for this notebook: the behaviour suite is
deterministic, so everyone gets the same table and can compare answers. Set a key in
`.env` when you want to measure real model behaviour.

In [ ]:
from research_agent.agent import AgentRunner
from research_agent.providers import MockModelProvider, OpenRouterProvider
from research_agent.tools import default_tool_registry

# LIVE comes from the course setup cell: True only when OPENROUTER_API_KEY was found.
provider = OpenRouterProvider() if LIVE else MockModelProvider()
tools = default_tool_registry()
runner = AgentRunner(provider, tools, max_steps=5)

print("Provider :", type(provider).__name__)
print("Tools    :", list(tools))
print("max_steps:", runner.max_steps)
print()
print("Mock results show that the APPLICATION works. Only a live run says anything")
print("about model quality - never report mock output as model-quality evidence.")

### Step 2 — Run the assistant once

One question that needs both tools. Read every field of the result: the status, how many
model turns it took, what it used, and what it cost.

In [ ]:
project_result = runner.run(
    "Explain what an AI agent is using local notes, then calculate 12 * 7."
)

print("status      :", project_result.status)
print("model turns :", project_result.steps)
print("usage       :", project_result.usage.model_dump())
print()
if project_result.response:
    print(project_result.response.model_dump_json(indent=2))
else:
    print("error:", project_result.error)

### Step 3 — The behaviour suite

Four cases, chosen so that each exercises a different path: no tool, one calculation, one
lookup, and both. For each one we record what we expected and what happened.

In [ ]:
cases = [
    {"id": "direct",      "question": "Give a brief greeting.",
     "expected_tools": []},
    {"id": "calculation", "question": "Calculate 12 * 7.",
     "expected_tools": ["calculator"]},
    {"id": "knowledge",   "question": "Use local notes to explain an AI tool.",
     "expected_tools": ["search_local_notes"]},
    {"id": "two_tools",   "question": "Use notes to explain an AI agent and calculate 12 * 7.",
     "expected_tools": ["calculator", "search_local_notes"]},
]

records = []
for case in cases:
    result = runner.run(case["question"])
    actual_tools = result.response.tools_used if result.response else []
    records.append({
        "case": case["id"],
        "status": result.status,
        "schema_valid": result.response is not None,
        "expected": case["expected_tools"],
        "actual": actual_tools,
        "tool_check": set(actual_tools) == set(case["expected_tools"]),
        "steps": result.steps,
        "tokens": result.usage.prompt_tokens + result.usage.completion_tokens,
        "cost_usd": round(result.usage.cost_usd, 6),
    })

header = f"{'case':12} {'status':10} {'schema':7} {'tools ok':9} {'steps':6} {'tokens':7} cost"
print(header)
print("-" * len(header))
for row in records:
    print(f"{row['case']:12} {row['status']:10} {str(row['schema_valid']):7} "
          f"{str(row['tool_check']):9} {row['steps']:<6} {row['tokens']:<7} {row['cost_usd']}")

passed = sum(row["tool_check"] and row["schema_valid"] for row in records)
print()
print(f"{passed} of {len(records)} cases behaved as expected.")

### Step 4 — Failure 1: an invalid tool argument

The model asks for the calculator with an argument our contract refuses. A tool returns a
`ToolResult` rather than raising, so the run continues and the model gets to read what went
wrong. Nothing here is a "hallucination" — it is a boundary doing its job.

In [ ]:
bad_arguments = [
    {"expression": ""},                                  # too short for the schema
    {"expression": "9" * 150},                           # too long for the schema
    {"wrong_key": "12 * 7"},                             # the required field is missing
    {"expression": "__import__('os').getcwd()"},         # shape fine, content refused
    {"expression": "1 / 0"},                             # valid arithmetic, invalid maths
]

calculator = tools["calculator"]
for index, arguments in enumerate(bad_arguments, start=1):
    outcome = calculator.execute(f"demo-{index}", arguments)
    shown = str(arguments)[:44]
    # Validation errors span several lines; squash them into one readable line.
    message = " ".join(outcome.output.split())
    print(f"{index}. {shown:46} is_error={outcome.is_error}")
    print(f"   -> {message[:100]}")

print()
print("Every one produced an observation the model can read. None raised, and none of")
print("them reached the arithmetic evaluator with data it had not approved.")

### Step 5 — Failure 2: a repeated tool request, stopped

A stuck model asks for the identical tool call over and over. Each request carries a
*new* id, so the runner cannot compare ids — it compares the tool name plus the arguments.
Without that check the run would quietly spend its whole step budget.

In [ ]:
from research_agent.schemas import Message, ModelTurn, ToolCall, ToolDefinition

class StuckProvider:
    """Always requests the same calculation, with a fresh call id every time."""

    def __init__(self):
        self.calls = 0

    def complete(self, messages: list[Message], tool_definitions: list[ToolDefinition]) -> ModelTurn:
        self.calls += 1
        return ModelTurn(tool_calls=[
            ToolCall(id=f"call-{self.calls}", name="calculator",
                     arguments={"expression": "2 + 2"})
        ])

stuck_provider = StuckProvider()
stuck_result = AgentRunner(stuck_provider, tools, max_steps=5).run("Calculate 2 + 2")

print("status              :", stuck_result.status)
print("error               :", stuck_result.error)
print("model turns used    :", stuck_result.steps, "of a possible 5")
print("provider calls made :", stuck_provider.calls)
print()
print("Stopped on the second identical request instead of burning all five turns.")
print("The signature is (tool name, arguments) - the call id is deliberately excluded,")
print("because a new id on every request would make every signature look unique.")

### Step 6 — Failure 3: invalid structured output

The last thing a model produces must satisfy `ResearchResponse`. Here the model answers in
friendly prose, which is exactly the failure mode 1.2 warned about. The run ends `failed`
with a specific message, and the partial trace is still available for debugging.

In [ ]:
class ChattyProvider:
    """Answers in prose instead of the JSON contract."""

    def complete(self, messages: list[Message], tool_definitions: list[ToolDefinition]) -> ModelTurn:
        return ModelTurn(content="Sure! An AI agent is a program that uses tools. Hope that helps!")

chatty_result = AgentRunner(ChattyProvider(), tools, max_steps=5).run("Explain an AI agent")

print("status  :", chatty_result.status)
print("response:", chatty_result.response)
print()
print("error (first 3 lines):")
for line in (chatty_result.error or "").splitlines()[:3]:
    print("   ", line)
print()
print("Messages kept for debugging:", [m.role for m in chatty_result.messages])

### Step 7 — Name the failing layer

When something goes wrong, resist the word "hallucination" and locate the layer instead.
Each row below maps to something you have now seen printed.

| What you observe          | Failing layer               | Where to look                        |
|---------------------------|-----------------------------|--------------------------------------|
| Wrong tool chosen         | model selection / prompting | the tool `description` text          |
| `Tool error: bad arguments` | model–schema boundary     | the argument model in `tools.py`     |
| `Tool error: <exception>` | Python execution            | the tool function itself             |
| `Final response failed validation` | output contract    | `ResearchResponse` and the system prompt |
| `Duplicate tool request stopped` | model looping        | tool descriptions, or the step budget |
| `max_steps`               | termination / control       | `AgentRunner(max_steps=...)`         |

### Try it yourself

Add a fifth behaviour case that you expect to *fail* the tool check, and confirm the suite
reports it rather than hiding it.

In [ ]:
# --- Worked solution ---
# A test suite is only useful if it can go red. We add a case whose expectation is
# deliberately wrong: the greeting needs no tool, but we claim it should use the
# calculator. A trustworthy suite must report False here.

failing_case = {"id": "expect_fail", "question": "Give a brief greeting.",
                "expected_tools": ["calculator"]}

result = runner.run(failing_case["question"])
actual_tools = result.response.tools_used if result.response else []

print("case          :", failing_case["id"])
print("expected tools:", failing_case["expected_tools"])
print("actual tools  :", actual_tools)
print("tool_check    :", set(actual_tools) == set(failing_case["expected_tools"]))
print("status        :", result.status, "(the RUN succeeded; the EXPECTATION did not)")
print()
print("Note the distinction: a completed run with the wrong tools is still a behaviour")
print("failure. Day 2 turns this idea into a proper golden set.")

### Checkpoint

**1. The suite reports `status=completed` for every case but `tool_check=False` for one of them. Is the project working?**

<details><summary>Show answer</summary>

Partly. `completed` only means the loop finished and the final answer satisfied the schema.
`tool_check` is about behaviour: did it use the tools we expected? A run can be perfectly
well-formed and still answer arithmetic from memory instead of calling the calculator.
Those are separate checks and the table keeps them separate on purpose.

</details>

**2. Why is a repeated tool request treated as a failure rather than just allowed to run again?**

<details><summary>Show answer</summary>

Because the identical call with identical arguments returns the identical observation, so
the model has learned nothing and the conversation is in a loop. Each repetition costs a
full model call. Stopping on the second one gives a clear error naming the tool, while
`max_steps` would eventually stop it too — later, more expensively, and with a vaguer
message.

</details>

### Recap

- **Limitation we saw:** one successful demo says nothing about behaviour, and a run can be
  well-formed and still wrong.
- **Layer we added:** a repeatable behaviour suite plus three deliberate failure
  demonstrations mapped to the layer that catches each one.
- **Evidence it worked:** four cases printed with status, schema validity, tool check,
  steps and cost; bad arguments produced observations instead of exceptions; the repeated
  request stopped after two turns; and prose output was rejected by the contract.